<a href="https://colab.research.google.com/github/VitorTardivo21/Redes-Neurais-e-IA-Aplicada/blob/main/01_exploracao_site.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pypdf requests pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 346.3/346.3 kB 2.0 MB/s eta 0:00:00


In [ ]:
"""
Etapa 1 — Coleta do Diário Oficial de Avaré
Baixa os PDFs do DOSP, extrai o texto por página e salva em CSV.
"""

import os
import re
import requests
import pandas as pd
import json
import base64

from datetime import datetime
from pypdf import PdfReader

# ==========================================
# LIGADURAS E LIMPEZA DE TEXTO
# ==========================================

LIGADURAS = {
    "ﬀ": "ff",
    "ﬁ": "fi",
    "ﬂ": "fl",
    "ﬃ": "ffi",
    "ﬄ": "ffl",
    "ﬅ": "st",
    "ﬆ": "st",
}

def limpar_texto(texto: str) -> str:
    """Remove ligaduras, normaliza espaços e elimina linhas duplicadas."""
    for ligadura, substituto in LIGADURAS.items():
        texto = texto.replace(ligadura, substituto)

    # Normaliza quebras de linha e espaços
    linhas = [linha.strip() for linha in texto.splitlines()]

    # Remove linhas duplicadas consecutivas
    linhas_unicas = []
    anterior = None
    for linha in linhas:
        if linha and linha != anterior:
            linhas_unicas.append(linha)
            anterior = linha

    # Junta com espaço único — sem \n no CSV
    return " ".join(linhas_unicas)

# ==========================================
# CONFIGURAÇÃO
# ==========================================

DATA_INICIAL = "2026-05-25"
DATA_FINAL   = "2026-05-31"

BASE_DIR     = os.path.dirname(os.path.abspath(__file__))
ARQUIVO_SAIDA = os.path.join(BASE_DIR, "diarios_avare.csv")
PASTA_PDFS    = os.path.join(BASE_DIR, "pdf")

os.makedirs(PASTA_PDFS, exist_ok=True)

# ==========================================
# API AVARÉ
# ==========================================

API_URL = "https://dosp.com.br/api/index.php/dioe.js/4700?callback=dioe"

HEADERS = {
    "User-Agent": "Mozilla/5.0"
}

sessao = requests.Session()
sessao.headers.update(HEADERS)

print("Baixando lista de edições...")

r = sessao.get(API_URL, timeout=60)

if r.status_code != 200:
    raise Exception(f"Erro ao acessar API ({r.status_code})")

# ==========================================
# CONVERTER JSONP → JSON
# ==========================================

texto = r.text
json_text = texto[texto.find("(") + 1 : texto.rfind(")")]
dados_api = json.loads(json_text)

print(f"Total de edições disponíveis: {len(dados_api['data'])}")

# ==========================================
# FILTRO DE DATAS
# ==========================================

data_inicial = datetime.strptime(DATA_INICIAL, "%Y-%m-%d")
data_final   = datetime.strptime(DATA_FINAL,   "%Y-%m-%d")

edicoes = []

for item in dados_api["data"]:
    try:
        data_edicao = datetime.strptime(item["data"][:10], "%Y-%m-%d")
        if data_inicial <= data_edicao <= data_final:
            edicoes.append(item)
    except Exception:
        pass

print(f"Edições encontradas no período: {len(edicoes)}")

# ==========================================
# DOWNLOAD E LEITURA DOS PDFS
# ==========================================

dados = []
pdfs_baixados   = 0
pdfs_existentes = 0

for posicao, item in enumerate(edicoes, start=1):

    try:
        iddo   = item["iddo"]
        codigo = base64.b64encode(str(iddo).encode()).decode()
        url_pdf = f"https://dosp.com.br/exibe_do.php?i={codigo}"

        print(f"[{posicao}/{len(edicoes)}] Edição {item['edicao_do']} ({item['data'][:10]})")

        # Organização por ano / mês
        data_pdf  = item["data"][:10]
        ano       = data_pdf[:4]
        mes       = data_pdf[5:7]
        pasta_ano_mes = os.path.join(PASTA_PDFS, ano, mes)
        os.makedirs(pasta_ano_mes, exist_ok=True)

        nome_pdf    = f"{data_pdf}_{item['edicao_do']}.pdf"
        caminho_pdf = os.path.join(pasta_ano_mes, nome_pdf)

        # Baixa somente se ainda não existir localmente
        if not os.path.exists(caminho_pdf):
            resposta_pdf = sessao.get(url_pdf, timeout=120)
            if resposta_pdf.status_code != 200:
                print("  Erro ao baixar PDF")
                continue
            with open(caminho_pdf, "wb") as arquivo_pdf:
                arquivo_pdf.write(resposta_pdf.content)
            pdfs_baixados += 1
            print(f"  PDF salvo: {nome_pdf}")
        else:
            pdfs_existentes += 1
            print(f"  PDF já existe: {nome_pdf}")

        # Extração de texto por página
        reader = PdfReader(caminho_pdf)

        for numero_pagina, pagina in enumerate(reader.pages, start=1):
            try:
                texto_pagina = pagina.extract_text() or ""
                texto_limpo  = limpar_texto(texto_pagina)
                if not texto_limpo:
                    continue

                dados.append({
                    "data":          item["data"][:10],
                    "edicao":        item["edicao_do"],
                    "pagina":        numero_pagina,
                    "paginas_total": item.get("pgtotal"),
                    "id":            f"{item['edicao_do']}_{numero_pagina}",
                    "url":           url_pdf,
                    "preview":       texto_limpo[:150],
                    "conteudo":      texto_limpo,
                })

            except Exception as erro_pagina:
                print(f"  Erro página {numero_pagina}: {erro_pagina}")

    except Exception as e:
        print(f"Erro na edição {item.get('edicao_do')}: {e}")

# ==========================================
# SALVAR CSV
# ==========================================

df = pd.DataFrame(dados)
df.to_csv(ARQUIVO_SAIDA, index=False, encoding="utf-8-sig")

# ==========================================
# RELATÓRIO FINAL
# ==========================================

print("\n=================================")
print("FINALIZADO")
print(f"Registros salvos : {len(df)}")
print(f"PDFs novos       : {pdfs_baixados}")
print(f"PDFs reaproveit. : {pdfs_existentes}")
print(f"Arquivo CSV      : {ARQUIVO_SAIDA}")
print(f"Pasta PDFs       : {PASTA_PDFS}")
print("=================================")

print("\nPrimeiras linhas:")
print(df.head())

NameError: name '__file__' is not defined